# Structured Output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the outpyt can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

So basically we are configuring the model to give the output in a certain schema that we have already designed.

## Pydantic
Pydantic models provide the richest feature set with field validation, description, and nested structures.

In [16]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.5)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    director:str = Field(description="The director of the movie")
    year:int = Field(description="The release year of the movie")
    rating:float = Field(description="The rating of the movie out of 10")

In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x10d5692b0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'strin

In [7]:
print(model_with_structure.invoke("Give me details of the movie 3 idiots in the form of a json"))

title='3 Idiots' director='Rajkumar Hirani' year=2009 rating=8.4


In [8]:
## Message output alongside structured output
## just add include_raw = true in the inoke method to get the raw message output along with the structured output

In [9]:
## Nested structure is also supported in pydantic models. You can have a list of another pydantic model as a field in your main model and langchain will be able to parse that as well.
class Actor(BaseModel):
    name:str = Field(description="The name of the actor")
    age:int = Field(description="The age of the actor")

class MovieDetailsWithActors(BaseModel):
    title:str = Field(description="The title of the movie")
    director:str = Field(description="The director of the movie")
    year:int = Field(description="The release year of the movie")
    rating:float = Field(description="The rating of the movie out of 10")
    actors:list[Actor] = Field(description="A list of actors in the movie")


model_with_nested_structure = model.with_structured_output(MovieDetailsWithActors)
print(model_with_nested_structure.invoke("Give me details of the movie 3 idiots in the form of a json along with the list of actors in the movie"))

title='3 Idiots' director='Rajkumar Hirani' year=2009 rating=8.4 actors=[Actor(name='Aamir Khan', age=44), Actor(name='Kareena Kapoor', age=29), Actor(name='R. Madhavan', age=34), Actor(name='Sharman Joshi', age=30), Actor(name='Omi Vaidya', age=33)]


## TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [17]:
from typing_extensions import TypedDict, Annotated

class MovieDetailsWithActorsTypedDict(TypedDict):
    title:str
    director:str
    year:int
    rating:float
    actors:list[Annotated[str, Field(description="The name of the actor")]]

model_with_typed_dict = model.with_structured_output(MovieDetailsWithActorsTypedDict)
model_with_typed_dict.invoke("Give me details of the movie 3 idiots in the form of a json along with the list of actors in the movie")

{'title': '3 Idiots',
 'director': 'Rajkumar Hirani',
 'year': 2009,
 'rating': 8.4,
 'actors': ['Aamir Khan',
  'R. Madhavan',
  'Sharman Joshi',
  'Kareena Kapoor',
  'Boman Irani']}

## DataClasses
A data class is a class typically containing mainly data, although there are not any restrictions. You create it using the @dataclass decorator

## Not implementing this for now 